## 🏠 import great_expectations as gx


In [ ]:
import great_expectations as gx
import pandas as pd
import os
import shutil
from great_expectations.checkpoint import Checkpoint

# --- CONFIGURACIÓN INICIAL ---

# 1. Configurar el Contexto de Datos
# GX 1.x utiliza un contexto para gestionar la configuración y los Data Docs.
context = gx.get_context()

# 2. Conectar a los Datos (Data Source y Asset)
# Definimos el origen de los datos para el archivo que limpiaste previamente.
datasource_name = "great_datasource"
datasource = context.data_sources.add_pandas(name=datasource_name)

# Agregamos el archivo limpio como un Asset de datos
asset_name = "data_set_great_expectations"
asset = datasource.add_csv_asset(
    name=asset_name, 
    filepath_or_buffer="data_set_great_expectations.csv"
)

batch_request = asset.build_batch_request()



# 2. Crear una Suite de Expectativas
expectation_suite_name = "ventas_expectations"
context.suites.add(gx.ExpectationSuite(name=expectation_suite_name))

# 3. Obtener un Validador para definir las pruebas
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)

# --- INICIO DE LAS PRUEBAS AUTOMATIZADAS (EXPECTATIONS) ---

# 1. Validación de Esquema
validator.expect_table_columns_to_match_ordered_list(
    column_list=['id_transaccion', 'id_producto', 'cantidad', 'precio_unitario', 'valor_total']
)

# 2. Validación de Rangos
validator.expect_column_values_to_be_between(
    column='cantidad',
    min_value=1,
    max_value=99,
    mostly=1.0  # Esperamos que el 100% de los valores cumplan
)

# 3. Validación de Unicidad
validator.expect_column_values_to_be_unique(column='id_transaccion')

# --- FIN DE LAS PRUEBAS ---

# Guardar las expectativas en la Suite
#validator.save_expectation_suite(discard_failed_expectations=False)

# --- EJECUTAR VALIDACIÓN Y GENERAR HTML ---
print("---------------------------------")
print(batch_request.dict)
print("---------------------------------")
print(validator.get_expectation_suite(expectation_suite_name))
print("---------------------------------")


'''
"data": {
        "datasource": {
        "name": "my_datasource",
        "id": "a758816-64c8-46cb-8f7e-03c12cea1d67"
    },
    "asset": {
        "name": "my_asset",
        "id": "b5s8816-64c8-46cb-8f7e-03c12cea1d67"
    },
    "batch_definition": {
        "name": "my_batch_definition",
        "id": "3a758816-64c8-46cb-8f7e-03c12cea1d67"
    }
'''

validation_def_name = "validacion_limpieza_ventas"
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name=validation_def_name,
        data=batch_request.dict,
        suite=validator.get_expectation_suite(expectation_suite_name)
    )
)

/home/idavid/miniconda3/envs/gx_env-13/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

/home/idavid/miniconda3/envs/gx_env-13/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

/home/idavid/miniconda3/envs/gx_env-13/lib/python3.13/site-packages/great_expectations/expectations/expectation.py:1659: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

---------------------------------
<bound method BatchRequest.dict of BatchRequest(datasource_name='great_datasource', data_asset_name='data_set_great_expectations', options={}, partitioner=None)>
---------------------------------
{
  "name": "ventas_expectations",
  "id": "a6c518d6-110a-4d60-a344-97fafa809c6e",
  "expectations": [
    {
      "type": "expect_table_columns_to_match_ordered_list",
      "kwargs": {
        "column_list": [
          "id_transaccion",
          "id_producto",
          "cantidad",
          "precio_unitario",
          "valor_total"
        ]
      },
      "meta": {},
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "cantidad",
        "min_value": 1.0,
        "max_value": 99.0
      },
      "meta": {},
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_unique",
      "kwargs": {
        "column": "id_transaccion"
      },
      "meta": {},

ValidationError: 1 validation error for ValidationDefinition
data
  Data must be a dictionary (if being deserialized) or a BatchDefinition object. (type=value_error)

In [ ]:
import great_expectations as gx

# 1. Get a Data Context
# This works with the "fluent" Python API
context = gx.get_context() 

# 2. Define a Data Source and Data Asset (e.g., a CSV file)
# The 'batching_regex' helps Great Expectations identify parts of the filename
# to use as 'options' for slicing the data.
datasource_name = "great_datasource"
datasource = context.data_sources.add_pandas(name=datasource_name)

# Agregamos el archivo limpio como un Asset de datos
asset_name = "data_set_great_expectations"
asset = datasource.add_csv_asset(
    name=asset_name, 
    filepath_or_buffer="data_set_great_expectations.csv"
)

# 3. Create a Batch Request to define a specific batch
# The 'options' dictionary specifies the exact slice of data you want to validate.
batch_request = asset.build_batch_request()
# This will target the file named "yellow_tripdata_sample_2019-02.csv".

# 4. Get the concrete Batch of data using the request
# This "instantiates" the batch, making the data available for validation
batch = context.get_batch(batch_request=batch_request)
